# 25 — Stochastic Local Volatility (SLV)

Heston + leverage function to match the local vol surface exactly.

- **Leverage calibration** (particle method)
- **SLV Monte Carlo** pricing
- Comparison: Heston vs SLV vs local vol
- Heston-HW hybrid (3-factor)

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ql_jax.models.equity.heston_slv import heston_slv_price_mc, calibrate_leverage_function
from ql_jax.engines.analytic.heston import heston_price
from ql_jax.engines.analytic.heston_hull_white import heston_hull_white_price
from ql_jax.processes.hybrid_heston_hw import HybridHestonHullWhiteProcess

S, r, q = 100.0, 0.03, 0.01
v0, kappa, theta, xi, rho = 0.04, 1.5, 0.04, 0.3, -0.7

---
## 1. Pure Heston Reference

In [ ]:
K, T = 100.0, 1.0
p_heston = float(heston_price(S, K, T, r, q, v0, kappa, theta, xi, rho, 1))
print(f"Heston ATM call: {p_heston:.6f}")

---
## 2. SLV with Trivial Leverage

In [ ]:
# Leverage = 1 everywhere => pure Heston MC
p_slv_trivial = float(heston_slv_price_mc(
    S, K, T, r, q, v0, kappa, theta, xi, rho,
    leverage_fn=1.0, option_type=1, n_paths=100_000, n_steps=200))

print(f"SLV MC (L=1, ≡ Heston) : {p_slv_trivial:.6f}")
print(f"Analytic Heston        : {p_heston:.6f}")
print(f"MC error               : {abs(p_slv_trivial - p_heston):.4f}")

---
## 3. Leverage Calibration

In [ ]:
# Simple flat local vol for demonstration
spots = jnp.linspace(60, 160, 20)
times = jnp.array([0.25, 0.5, 1.0, 2.0])
local_vols = jnp.full((len(times), len(spots)), 0.20)  # flat 20% local vol

leverage = calibrate_leverage_function(
    spots, times, local_vols,
    v0, kappa, theta, xi, rho,
    n_mc_paths=5000)

print(f"Leverage shape: {leverage.shape}")
print(f"Leverage range: [{float(leverage.min()):.4f}, {float(leverage.max()):.4f}]")

plt.figure(figsize=(8, 5))
for i, t in enumerate(times):
    plt.plot(np.array(spots), np.array(leverage[i]), label=f't={float(t):.2f}')
plt.xlabel('Spot')
plt.ylabel('Leverage L(t,S)')
plt.title('Calibrated Leverage Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 4. Heston-Hull-White Hybrid

In [ ]:
A_HW, SIGMA_HW = 0.03, 0.01

disc_fn = lambda t: jnp.exp(-r * t)

for rho_sr in [-0.3, 0.0, 0.3]:
    p = float(heston_hull_white_price(
        S, K, T, r, q, v0, kappa, theta, xi, rho,
        A_HW, SIGMA_HW, rho_sr, rho_vr=0.0,
        option_type=1, discount_curve_fn=disc_fn))
    print(f"  Heston-HW (ρ_sr={rho_sr:+.1f}) : {p:.6f}")

print(f"  Pure Heston           : {p_heston:.6f}")

---
## 5. Hybrid Process Simulation

In [ ]:
hybrid = HybridHestonHullWhiteProcess(
    S0=S, v0=v0, r0=r, q=q,
    kappa=kappa, theta=theta, xi=xi, rho_sv=rho,
    a=A_HW, sigma_r=SIGMA_HW, rho_sr=0.1)

key = jax.random.PRNGKey(42)
S_paths, v_paths, r_paths = hybrid.simulate(T, n_steps=100, n_paths=5000, key=key)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(np.linspace(0, T, 101), np.array(S_paths[:, :20]))
axes[0].set_title('Stock Paths')
axes[0].set_xlabel('Time')

axes[1].plot(np.linspace(0, T, 101), np.array(v_paths[:, :20]))
axes[1].set_title('Variance Paths')
axes[1].set_xlabel('Time')

axes[2].plot(np.linspace(0, T, 101), np.array(r_paths[:, :20]))
axes[2].set_title('Short Rate Paths')
axes[2].set_xlabel('Time')

fig.tight_layout()
plt.show()

---
## 6. Exercises

1. **Market smile matching**: Calibrate leverage to reproduce a market-like vol smile with negative skew.
2. **Barrier under SLV**: Price a barrier option under SLV and compare to pure Heston.
3. **3-factor Greeks**: Compute dV/dρ_sr for the Heston-HW model using AD.

---
*End of Notebook 25*